# Replicate draws on Colab - GRPO_LA5 @10 and PTO_LA0 @10

The two **replicate** generate-only passes (STATUS.md section 'replicate draw'), on an
**A100 runtime**. Run All, top to bottom - cell 1 mounts Drive and preflights every path,
so nothing below can fail on a missing file.

- **Batch size 64** (each arm's own stored `conversation_batch_size`, and what the primary
  `model_iter_10` draws used on this same hardware). 96 convs = 2 rounds. See cell 3.
- Output lands in `conversations/replicate/<EXP>/model_iter_10_rep1_TT0.9_TP0.7/` -
  **outside** `conversations/full/`, so auto-discovery never sees it (deliberate; the score
  lake names carry a `_rep1_` infix instead). Resume-safe per conversation CSV.
- Keys come from Colab Secrets (`OPENAI_API_KEY`, `huggingface`), like the trainer notebooks.
- **No installs by default** (the recipe the Aug-25 trainer runs used). If you uncomment the
  pins in cell 2, **Runtime -> Restart session afterwards, then Run All again** - cell 2's
  guard enforces this instead of letting a mixed numpy explode later.
- The `seed + k + 1` shuffle convention was verified against both arms with `--verify-seeds`
  on 2026-08-26 (726/717 correct, 0 wrong, both decoy offsets fail), so it is not re-run here.
- Afterwards: let Drive Desktop finish syncing, then locally run
  `eda/tools/score_replicate.py --primary --judge`, then `eda/tools/replicate_check.py`.

In [ ]:
# === 1. Mount Drive, then check every path this notebook needs =========================
# Drive MUST be mounted before any /content/drive/... path is touched - including by the
# %run lines below, which have to FIND the script before they can execute it. (The script
# mounts Drive itself, but that is too late to help %run locate it.)
import os, sys

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')          # idempotent: a no-op if already mounted
else:
    print('Not on Colab - this notebook is written for a Colab A100 runtime.')

ROOT = '/content/drive/MyDrive/Thesis_PTO_GRPO/Exp3_PTO_GRPO'
SCRIPT = '/content/drive/MyDrive/Thesis_PTO_GRPO/Exp3_PTO_GRPO/code/tools/generate_eval_convs.py'

checks = [
    ('generate tool', SCRIPT),
    ('GRPO run_metadata', ROOT + '/data/grpo_Exp3/runs/full/GRPO_Iterative_Q1Q2_Llama32-1B_LA5_MCL12_G8/run_metadata.json'),
    ('GRPO iter-10 adapter', ROOT + '/data/grpo_Exp3/runs/full/GRPO_Iterative_Q1Q2_Llama32-1B_LA5_MCL12_G8/iteration_10/adapter'),
    ('PTO run_metadata', ROOT + '/data/pto_Exp3/runs/full/PTO_Iterative_Q1Q2_Llama32-1B_LA0_MCL12_M8_PTgreedy/run_metadata.json'),
    ('PTO iter-10 adapter', ROOT + '/data/pto_Exp3/runs/full/PTO_Iterative_Q1Q2_Llama32-1B_LA0_MCL12_M8_PTgreedy/iteration_10/adapter'),
    ('_shared package', ROOT + '/code/_shared'),
    ('grpo_trainer', ROOT + '/code/GRPO_Exp3/grpo_trainer.py'),
    ('pto_trainer', ROOT + '/code/PTO_Exp3/pto_trainer.py'),
]
missing = [name for name, p in checks if not os.path.exists(p)]
for name, p in checks:
    print(('  OK   ' if os.path.exists(p) else '  MISSING '), name, '->', p)

# The tool must be the --method version (added 2026-08-26); an older copy on Drive would
# fail later with 'unrecognized arguments: --method', so catch it here instead.
if os.path.exists(SCRIPT):
    has_method = '--method' in open(SCRIPT, encoding='utf-8').read()
    print(('  OK    ' if has_method else '  STALE '), 'generate tool supports --method')
    if not has_method:
        missing.append('generate tool is an old copy (no --method) - re-sync code/ to Drive')

try:
    from google.colab import userdata
    for k in ('OPENAI_API_KEY', 'huggingface'):
        try:
            userdata.get(k)
            print('  OK    secret', k)
        except Exception as e:
            print('  MISSING secret', k, '->', type(e).__name__)
            missing.append('Colab secret ' + k)
except ImportError:
    pass

if missing:
    raise SystemExit('PREFLIGHT FAILED: ' + '; '.join(missing))
print()
print('Preflight OK - run the cells below in order.')

In [ ]:
# === 2. Colab environment =============================================================
# GROUND TRUTH: the trainer notebook that ran GRPO iterations 7-10 on Colab (2026-08-20..25)
# has this ENTIRE block commented out - stock Colab, no installs, was the proven recipe.
# So: leave the pins commented unless imports actually fail in cell 4.
#
# If you DO uncomment and run the install: you MUST restart afterwards
# (Runtime -> Restart session), then Run All again. Colab's kernel imports numpy/pandas at
# startup for its own integrations, so an install that changes numpy leaves the old compiled
# core cached in the running process while new .py files sit on disk - the mix explodes as
# ImportError: cannot import name '_slice' from 'numpy._core.umath' (seen 2026-08-26).
# Installs persist on the VM across a session restart, so the second Run All no-ops here.
#
# %pip install -q accelerate==1.13.0 bitsandbytes==0.49.2 datasets==4.8.5 \
#              huggingface_hub==1.14.0 numpy==2.4.4 openai==2.36.0 pandas==3.0.3 \
#              peft==0.19.1 scipy==1.17.1 sentence-transformers==5.5.0 \
#              tensorboard==2.20.0 transformers==5.8.1 trl==1.4.0 wandb==0.26.1

# torchao: Colab pre-bakes < 0.16.0, which the PINNED peft 0.19.1 rejects by raising inside
# its LoRA dispatcher (runs for PeftModel.from_pretrained too, so this no-train pass hits
# it). Needed whenever the pins above were installed; harmless a no-op on stock Colab.
%pip uninstall -y -q torchao

# --- Skew guard: catches the restart-needed state instead of a cryptic ImportError later.
import importlib.metadata as _md
import sys

_disk = _md.version('numpy')
_loaded = sys.modules['numpy'].__version__ if 'numpy' in sys.modules else None
print(f'numpy: on disk {_disk}, loaded in kernel {_loaded or "(not imported yet)"}')
if _loaded and _loaded != _disk:
    raise SystemExit(
        f'RESTART NEEDED: numpy {_loaded} is loaded in this kernel but {_disk} is on disk\n'
        '(a package install changed it this session). Runtime -> Restart session, then\n'
        'Run All again - installs persist on the VM, so this passes on the second run.')
print('Environment OK.')

In [ ]:
# === 3. Which GPU did Colab assign, and does the stored batch fit? =====================
# Both arms store conversation_batch_size = 64 (verified from run_metadata.json), and that
# is what the passes below use - the SAME value the primary model_iter_10 draws were
# generated at, so batch composition is not a variable in the comparison they feed.
#
# Budget line (CLAUDE.md 'Gotchas', measured): 2.6 GB weights + ~1.1 GB per concurrent conv.
#   batch 64 -> 2.6 + 64*1.1 = 73.0 GB   (fits an 80 GB A100, ~91% of it)
#   batch 96 -> 2.6 + 96*1.1 = 108.2 GB  (does NOT fit; would be one round instead of two)
# At batch 64 on an A100 the pass is API-BOUND, not VRAM-bound (patient_api_concurrency=96,
# every turn waits on gpt-4o-mini), so filling VRAM further buys wall-clock, not throughput.
import subprocess

print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                      '--format=csv,noheader'], capture_output=True, text=True).stdout.strip())
print()
print('If that says ~40 GB rather than ~80 GB, add  --batch-size 32  to the two run cells')
print('below (2.6 + 32*1.1 = 37.8 GB). It is a throughput knob only: batching does not')
print('change the sampled conversations, so a 40 GB run is still valid replicate data.')

In [ ]:
# === 4. Draw 1: GRPO K=5, iteration 10 =================================================
# The best final state on both graders - and the state with the worst per-conversation
# cross-judge agreement in the experiment, which is why it is the one that gets replicated.
# ~15-30 min. Resume-safe: re-running skips conversations already written.
%run /content/drive/MyDrive/Thesis_PTO_GRPO/Exp3_PTO_GRPO/code/tools/generate_eval_convs.py --method grpo --iter 10 --experiment GRPO_Iterative_Q1Q2_Llama32-1B_LA5_MCL12_G8 --conv-dir /content/drive/MyDrive/Thesis_PTO_GRPO/Exp3_PTO_GRPO/data/grpo_Exp3/conversations/replicate/GRPO_Iterative_Q1Q2_Llama32-1B_LA5_MCL12_G8/model_iter_10_rep1_TT0.9_TP0.7

In [ ]:
# === 5. Draw 2: PTO K=0, iteration 10 ==================================================
# The arm GRPO K=5 beats by only 0.007 on the held-out judge - the tie the replicate is
# meant to test. ~15-30 min.
%run /content/drive/MyDrive/Thesis_PTO_GRPO/Exp3_PTO_GRPO/code/tools/generate_eval_convs.py --method pto --iter 10 --experiment PTO_Iterative_Q1Q2_Llama32-1B_LA0_MCL12_M8_PTgreedy --conv-dir /content/drive/MyDrive/Thesis_PTO_GRPO/Exp3_PTO_GRPO/data/pto_Exp3/conversations/replicate/PTO_Iterative_Q1Q2_Llama32-1B_LA0_MCL12_M8_PTgreedy/model_iter_10_rep1_TT0.9_TP0.7

In [ ]:
# === 6. Both draws complete? ===========================================================
import os

dirs = ['/content/drive/MyDrive/Thesis_PTO_GRPO/Exp3_PTO_GRPO/data/grpo_Exp3/conversations/replicate/GRPO_Iterative_Q1Q2_Llama32-1B_LA5_MCL12_G8/model_iter_10_rep1_TT0.9_TP0.7',
        '/content/drive/MyDrive/Thesis_PTO_GRPO/Exp3_PTO_GRPO/data/pto_Exp3/conversations/replicate/PTO_Iterative_Q1Q2_Llama32-1B_LA0_MCL12_M8_PTgreedy/model_iter_10_rep1_TT0.9_TP0.7']
ok = True
for d in dirs:
    n = len([f for f in os.listdir(d) if f.startswith('conversation_')]) if os.path.isdir(d) else 0
    ok &= n >= 96
    print(('OK   ' if n >= 96 else 'SHORT'), n, '/96 -', d.split('/data/')[-1])
print()
print('Both at 96/96 - let Drive Desktop finish syncing (tray tick), then run locally:'
      if ok else 'Not finished - re-run the cell above; it resumes where it stopped.')
if ok:
    print('  .venv/Scripts/python.exe Exp3_PTO_GRPO/eda/tools/score_replicate.py --primary --judge')
    print('  .venv/Scripts/python.exe Exp3_PTO_GRPO/eda/tools/replicate_check.py')